In [ ]:
print('===== NATIVE SOFTMAX CAUSAL BWD BITEXACT TEST =====')
print('===== NATIVE SOFTMAX CAUSAL FWD BITEXACT TEST =====')
print('===== NATIVE MERGEHBHEAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/softmax_cb_cpu.cpu.c'; so=ex+'/softmax_cb_cpu.so'
# inject exact SoftmaxCausalBwd C from tkv_bwd_ops.c
cc_path=ex+'/softmax_cb_cpu.cpu.c'
txt=open(cc_path).read()
cb=r"""
static inline void tkv_softmax_cb(float* dp, const float* P, const float* dctx,
                                  const float* ctx, int n, int hd) {
    for (int t = 0; t < n; t++) {
        const float* d  = dctx + (long long)t * hd;
        const float* ct = ctx  + (long long)t * hd;
        const float* p  = P    + (long long)t * n;
        float*       ds = dp   + (long long)t * n;
        double dotv = 0.0;
        for (int u = 0; u < hd; u++) {
            float prod = d[u] * ct[u];
            dotv = dotv + (double)prod;
        }
        for (int u = 0; u <= t; u++) ds[u] = (float)((double)p[u] * ((double)ds[u] - dotv));
        for (int u = t + 1; u < n; u++) ds[u] = 0.0f;
    }
}
"""
txt=txt.replace('#include <math.h>', '#include <math.h>\n'+cb)
open(cc_path,'w').write(txt)
print('softmax BWD C injected =', 'tkv_softmax_cb' in open(cc_path).read())
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare SoftmaxCausalBwd
import ctypes, numpy as np
args=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,
      ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_int,ctypes.c_int]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').SoftmaxCausalBwd
native.argtypes=args; native.restype=ctypes.c_int
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/softmax_cb_cpu.so').SoftmaxCausalBwd
tkv.argtypes=args; tkv.restype=ctypes.c_int
rng=np.random.default_rng(7)
for n,hd in [(8,16),(16,64),(64,128),(127,64)]:
    dp0=rng.standard_normal(n*n).astype(np.float32); P=rng.random((n,n)).astype(np.float32)
    dctx=rng.standard_normal(n*hd).astype(np.float32); ctx=rng.standard_normal(n*hd).astype(np.float32)
    a=dp0.copy(); c=dp0.copy()
    pa=[x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)) for x in [a,P,dctx,ctx]]
    pc=[x.ctypes.data_as(ctypes.POINTER(ctypes.c_float)) for x in [c,P,dctx,ctx]]
    native(*(pa+[a.size,P.size,dctx.size,ctx.size,n,hd]))
    tkv(*(pc+[c.size,P.size,dctx.size,ctx.size,n,hd]))
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('n=%d hd=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(n,hd,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')